# Importing libraries and data

In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
df = pd.read_parquet("parquets/dati_estrazioni_ricoveri.parquet")

# Overview on all the data

In [3]:
df

,PSEUDO_PAZ,ANNO,DATA_APERTURA,DATA_CHIUSURA,MODALITA_DIMISSIONE_COD,STRUTTURA_COD,TIPO_ATTIVITA_COD,SESSO_COD,PROVENIENZA_COD,ISTRUZIONE_COD,...,'S03','S04','INTERVENTO_P','INTERVENTO_S01','INTERVENTO_S02','INTERVENTO_S03','INTERVENTO_S04','INTERVENTO_S',COMPILAZIONE_CHECKLIST_COD,"LISTAGG(C.DATA_INTERVENTO,',')OVER(PARTITIONBYA.NOSOLOGICO,A.ANNO,A.STRUTTURA_COD)"
2,20525758880092952616734,2022,2022-08-01 15:47:00,2022-08-10 14:30:00,2,42006,RI,1,2,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"01-AGO-22,01-AGO-22,01-AGO-22,01-AGO-22"
3,2719961031659409949680,2020,2020-12-08 16:09:00,2020-12-15 16:30:00,2,42006,RI,1,5,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"08-DIC-20,08-DIC-20,08-DIC-20,08-DIC-20,08-DIC..."
4,30961298531313452439580,2022,2022-01-17 20:10:00,2022-01-21 13:10:00,2,42006,RI,1,2,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"17-GEN-22,17-GEN-22,17-GEN-22,17-GEN-22"
5,2719961031659409949680,2020,2020-12-08 16:09:00,2020-12-15 16:30:00,2,42006,RI,1,5,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"08-DIC-20,08-DIC-20,08-DIC-20,08-DIC-20,08-DIC..."
6,5712560221500663468891,2001,2001-01-17 08:00:00,2001-06-08 00:00:00,2,42006,RI,1,1,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1380,5443382870799330932690,2022,2022-08-29 03:44:00,2022-09-21 15:00:00,8,42001,RI,1,5,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"29-AGO-22,30-AGO-22,30-AGO-22,30-AGO-22,30-AGO..."
1381,8688775490799534962197,2018,2018-05-22 08:00:00,2018-05-29 13:00:00,8,42006,RI,1,3,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"25-MAG-18,23-MAG-18"
1382,13280116591091443765758,2017,2017-03-06 08:15:00,2017-03-15 12:00:00,8,42006,RI,1,4,2,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"09-MAR-17,09-MAR-17,09-MAR-17,09-MAR-17,09-MAR..."
1383,2462337189602073236450,2022,2022-03-29 13:09:00,2022-04-26 18:20:00,9,42010,RI,2,5,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,30-MAR-22


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1383 entries, 2 to 1384
Data columns (total 46 columns):
 #   Column                                                                              Non-Null Count  Dtype         
---  ------                                                                              --------------  -----         
 0   PSEUDO_PAZ                                                                          1383 non-null   string        
 1   ANNO                                                                                1383 non-null   Int64         
 2   DATA_APERTURA                                                                       1383 non-null   datetime64[us]
 3   DATA_CHIUSURA                                                                       1383 non-null   datetime64[us]
 4   MODALITA_DIMISSIONE_COD                                                             1383 non-null   string        
 5   STRUTTURA_COD                                              

# Analysis

### Checks on specific values

In [86]:
now_column = 'DATA_PRENOTAZIONE'

# # To exclude time
# df[now_column] = df[now_column].dt.normalize()

# # To exclude date
# df[now_column] = df[now_column].dt.time

# Total number of rows: 
df[now_column]

2             NaT
3             NaT
4             NaT
5             NaT
6             NaT
          ...    
1380          NaT
1381   2018-03-23
1382   2016-12-13
1383          NaT
1384          NaT
Name: DATA_PRENOTAZIONE, Length: 1383, dtype: datetime64[us]

### Overview on all the types found on the column

In [87]:
df[now_column].apply(type).value_counts()

DATA_PRENOTAZIONE
<class 'pandas.api.typing.NaTType'>    697
<class 'pandas.Timestamp'>             686
Name: count, dtype: int64

### Get a list of all the values, uniqued

In [88]:
df[now_column].unique().tolist()

[NaT,
 Timestamp('2019-12-21 00:00:00'),
 Timestamp('2020-11-12 00:00:00'),
 Timestamp('2025-05-07 00:00:00'),
 Timestamp('2024-07-02 00:00:00'),
 Timestamp('2021-05-25 00:00:00'),
 Timestamp('2024-11-19 00:00:00'),
 Timestamp('2022-10-23 00:00:00'),
 Timestamp('2021-12-06 00:00:00'),
 Timestamp('2024-11-28 00:00:00'),
 Timestamp('2023-01-24 00:00:00'),
 Timestamp('2022-12-07 00:00:00'),
 Timestamp('2021-02-03 00:00:00'),
 Timestamp('2025-05-15 00:00:00'),
 Timestamp('2025-01-09 00:00:00'),
 Timestamp('2024-02-28 00:00:00'),
 Timestamp('2019-12-20 00:00:00'),
 Timestamp('2019-06-20 00:00:00'),
 Timestamp('2024-07-08 00:00:00'),
 Timestamp('2025-10-23 00:00:00'),
 Timestamp('2018-10-25 00:00:00'),
 Timestamp('2020-01-28 00:00:00'),
 Timestamp('2025-08-08 00:00:00'),
 Timestamp('2019-05-28 00:00:00'),
 Timestamp('2022-09-07 00:00:00'),
 Timestamp('2024-12-20 00:00:00'),
 Timestamp('2019-03-21 00:00:00'),
 Timestamp('2024-01-02 00:00:00'),
 Timestamp('2025-05-06 00:00:00'),
 Timestamp('20

### Get how many times a unique value is found in the column, ordered from more to less frequent

In [89]:
df[now_column].value_counts(dropna=False).to_dict()

{NaT: 697,
 Timestamp('2025-05-07 00:00:00'): 6,
 Timestamp('2021-05-25 00:00:00'): 6,
 Timestamp('2021-02-03 00:00:00'): 6,
 Timestamp('2018-11-29 00:00:00'): 6,
 Timestamp('2025-08-08 00:00:00'): 5,
 Timestamp('2024-11-19 00:00:00'): 4,
 Timestamp('2025-01-09 00:00:00'): 4,
 Timestamp('2024-07-08 00:00:00'): 4,
 Timestamp('2024-12-20 00:00:00'): 4,
 Timestamp('2020-12-17 00:00:00'): 4,
 Timestamp('2025-09-25 00:00:00'): 4,
 Timestamp('2025-03-07 00:00:00'): 4,
 Timestamp('2019-11-26 00:00:00'): 4,
 Timestamp('2025-05-27 00:00:00'): 4,
 Timestamp('2024-09-06 00:00:00'): 4,
 Timestamp('2018-10-16 00:00:00'): 4,
 Timestamp('2018-12-11 00:00:00'): 4,
 Timestamp('2016-12-13 00:00:00'): 4,
 Timestamp('2018-03-23 00:00:00'): 4,
 Timestamp('2017-03-22 00:00:00'): 3,
 Timestamp('2024-06-20 00:00:00'): 3,
 Timestamp('2017-07-19 00:00:00'): 3,
 Timestamp('2022-10-04 00:00:00'): 3,
 Timestamp('2019-03-18 00:00:00'): 3,
 Timestamp('2019-09-03 00:00:00'): 3,
 Timestamp('2024-04-29 00:00:00'): 3,
 

### Get the number of how many different values there are

In [90]:
len(df[now_column].unique().tolist())

489

In [91]:
df[now_column].value_counts(dropna=False).shape

(489,)

### Detect type inconsistencies for numeric values (dataframe has to be EMPTY)

In [92]:
temp = pd.to_numeric(df[now_column], errors='coerce').isna()
temp = df[temp]
temp

,PSEUDO_PAZ,ANNO,DATA_APERTURA,DATA_CHIUSURA,MODALITA_DIMISSIONE_COD,STRUTTURA_COD,TIPO_ATTIVITA_COD,SESSO_COD,PROVENIENZA_COD,ISTRUZIONE_COD,...,'S03','S04','INTERVENTO_P','INTERVENTO_S01','INTERVENTO_S02','INTERVENTO_S03','INTERVENTO_S04','INTERVENTO_S',COMPILAZIONE_CHECKLIST_COD,"LISTAGG(C.DATA_INTERVENTO,',')OVER(PARTITIONBYA.NOSOLOGICO,A.ANNO,A.STRUTTURA_COD)"


### Consistency among columns

#### Check perfect consistency among two columns
Returns an empty df if there is consistency

In [93]:
# column1 = 'AIC_COD'
# column2 = 'AIC_qty'

# inconsistent = df.groupby(column1)[column2].nunique() > 1
# problem_keys = inconsistent[inconsistent].index
# df[df[column1].isin(problem_keys)]

#### Check if some _specific_ values are consinstent across columns
Returns `True` if there is consistency

In [94]:
# column1 = 'AIC_COD'
# value1 = '026966046'
# column2 = 'AIC_COD'
# value2 = '026966046'

# mask = df[column1] == value1
# print(mask.any() and (df.loc[mask, column2] == value2).all()) # For normal values
# # print(mask.any() and (df.loc[mask, column2].isna()).all()) # For NaNs

### Get a specific row value and informations about it

In [95]:
value = df.loc[
    42, # Row number
    'BRANCA_COD'# Column    
]

print(value)
print(type(value))
print(pd.isna(value))

KeyError: 'BRANCA_COD'